In [1]:
!pip install segmentation_models_pytorch
import segmentation_models_pytorch as smp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.0 MB/s eta 0:00:00


In [2]:
import ee
import geemap
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
ee.Authenticate()
ee.Initialize(project='coringa-mangrove')

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix, classification_report



In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [6]:
CORINGA_BBOX = [82.13, 16.42, 82.23, 17.0]
REGION = ee.Geometry.Rectangle(CORINGA_BBOX)

START_DATE = '2022-09-01'
END_DATE = '2022-12-31'

PATCH_SIZE = 128
STRIDE = 64

BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-4

INPUT_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
INDEX_BANDS = ['NDVI', 'NDMI', 'MNDWI', 'GCVI', 'SR']
ALL_FEATURES = INPUT_BANDS + INDEX_BANDS
N_CHANNELS = len(ALL_FEATURES)

MANGROVE_CLASS = 95

In [7]:
def load_sentinel2():
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(REGION)
          .filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
          .median()
          .clip(REGION))
    return s2

def compute_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
    mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    gcvi = image.expression('B8 / B3 - 1', {
        'B8': image.select('B8'), 'B3': image.select('B3')
    }).rename('GCVI')
    sr = image.expression('B8 / B4', {
        'B8': image.select('B8'), 'B4': image.select('B4')
    }).rename('SR')
    return image.addBands([ndvi, ndmi, mndwi, gcvi, sr])

def prepare_features():
    s2 = load_sentinel2()
    s2_bands = s2.select(INPUT_BANDS)
    with_indices = compute_indices(s2_bands)
    return with_indices.select(ALL_FEATURES)

print('Loading Sentinel-2 imagery...')
features_image = prepare_features()
print('Feature bands:', features_image.bandNames().getInfo())

Loading Sentinel-2 imagery...
Feature bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDMI', 'MNDWI', 'GCVI', 'SR']


In [8]:
def load_worldcover():
    wc = (ee.ImageCollection('ESA/WorldCover/v200/2021')
          .filterBounds(REGION)
          .first()
          .clip(REGION))
    mangrove_mask = wc.eq(MANGROVE_CLASS).rename('mangrove')
    return mangrove_mask

print('Loading WorldCover labels...')
labels_image = load_worldcover()
print('Labels loaded.')

Loading WorldCover labels...
Labels loaded.


In [9]:
def extract_patches(features_img, labels_img, patch_size=128, n_patches=1000):
    region_bounds = REGION.bounds().getInfo()['coordinates'][0]
    min_lon, min_lat = region_bounds[0]
    max_lon, max_lat = region_bounds[2]

    np.random.seed(42)
    lons = np.random.uniform(min_lon, max_lon, n_patches)
    lats = np.random.uniform(min_lat, max_lat, n_patches)

    X_list = []
    y_list = []
    scale = 10
    half_patch = patch_size // 2

    for i, (lon, lat) in enumerate(zip(lons, lats)):
        if i % 50 == 0:
            print(f'  Patch {i+1}/{n_patches}...')

        center = ee.Geometry.Point([lon, lat])
        buffer = center.buffer(half_patch * scale).bounds()

        try:
            feat_patch = features_img.sampleRegion(
                collection=ee.FeatureCollection([ee.Feature(buffer)]),
                scale=scale,
                geometries=False
            ).getInfo()

            lbl_patch = labels_img.sampleRegion(
                collection=ee.FeatureCollection([ee.Feature(buffer)]),
                scale=scale,
                geometries=False
            ).getInfo()

            if not feat_patch.get('features'):
                continue

            feat_props = feat_patch['features'][0].get('properties', {})
            lbl_props = lbl_patch['features'][0].get('properties', {})

            if not feat_props or not lbl_props:
                continue

            n_pixels = patch_size * patch_size

            feat_list = []
            bad = False
            for b in ALL_FEATURES:
                if b not in feat_props:
                    bad = True
                    break
                val = feat_props[b]
                if not isinstance(val, list) or len(val) != n_pixels:
                    bad = True
                    break
                feat_list.append(val)

            if bad:
                continue

            feat_arr = np.array(feat_list, dtype=np.float32).reshape(N_CHANNELS, patch_size, patch_size)

            lbl_val = lbl_props.get('mangrove')
            if not isinstance(lbl_val, list) or len(lbl_val) != n_pixels:
                continue
            lbl_arr = np.array(lbl_val, dtype=np.float32).reshape(1, patch_size, patch_size)

            X_list.append(feat_arr)
            y_list.append(lbl_arr)

        except:
            continue

    if len(X_list) == 0:
        print('ERROR: No patches extracted. Check GEE auth and region bounds.')
        return None, None

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    for c in range(X.shape[1]):
        mean = X[:, c].mean()
        std = X[:, c].std() + 1e-8
        X[:, c] = (X[:, c] - mean) / std

    print(f'Extracted {X.shape[0]} patches')
    return X, y

X, y = extract_patches(features_image, labels_image, PATCH_SIZE, n_patches=1000)
if X is None:
    print('Stopping. Fix the issue above first.')
else:
    print(f'X: {X.shape}, y: {y.shape}')
    print(f'Mangrove ratio: {y.mean():.4f}')

  Patch 1/1000...
  Patch 51/1000...
  Patch 101/1000...
  Patch 151/1000...
  Patch 201/1000...
  Patch 251/1000...
  Patch 301/1000...
  Patch 351/1000...
  Patch 401/1000...
  Patch 451/1000...
  Patch 501/1000...
  Patch 551/1000...
  Patch 601/1000...
  Patch 651/1000...
  Patch 701/1000...
  Patch 751/1000...
  Patch 801/1000...
  Patch 851/1000...
  Patch 901/1000...
  Patch 951/1000...
ERROR: No patches extracted. Check GEE auth and region bounds.
Stopping. Fix the issue above first.


In [10]:
# Debug: see what GEE actually returns
center = ee.Geometry.Point([82.18, 16.7])
buffer = center.buffer(640).bounds()

test_feat = features_image.sample(
    region=buffer,
    scale=10,
    numPixels=1,
    geometries=False
).getInfo()

print('Keys:', test_feat.keys())
print('Number of features:', len(test_feat['features']))
print('Properties keys:', list(test_feat['features'][0]['properties'].keys()))
for k, v in test_feat['features'][0]['properties'].items():
    if isinstance(v, list):
        print(f'  {k}: list of len {len(v)}')
    else:
        print(f'  {k}: {type(v).__name__} = {v}')

Keys: dict_keys(['type', 'columns', 'properties', 'features'])
Number of features: 1
Properties keys: ['B11', 'B12', 'B2', 'B3', 'B4', 'B8', 'GCVI', 'MNDWI', 'NDMI', 'NDVI', 'SR']
  B11: int = 1454
  B12: int = 809
  B2: int = 468
  B3: int = 658
  B4: int = 522
  B8: int = 2020
  GCVI: float = 2.069908814589666
  MNDWI: float = -0.3768939393939394
  NDMI: float = 0.1629245826137018
  NDVI: float = 0.5892997639653816
  SR: float = 3.8697318007662833


In [11]:
# Install geemap (already done) and use ee_to_numpy
import geemap

def image_to_numpy(image, region, scale=10):
    """Convert GEE image to numpy array."""
    arr = geemap.ee_to_numpy(image, region=region, scale=scale)
    return arr

print('Converting image to numpy...')
full_array = image_to_numpy(features_image, REGION, scale=50)
print(f'Full array shape: {full_array.shape}')

Converting image to numpy...
Full array shape: (1292, 224, 11)


In [12]:
wc_raw = ee.Image('ESA/WorldCover/v200/2021')
labels_raw = wc_raw.eq(MANGROVE_CLASS).rename('mangrove').clip(REGION)

# Add to features
combined = features_image.addBands(labels_raw)

print('Downloading combined image...')
combined_array = geemap.ee_to_numpy(combined, region=REGION, scale=50)
print(f'Combined shape: {combined_array.shape}')

Combined shape: (1292, 224, 12)


In [13]:
# Split features and labels
features_array = combined_array[:, :, :11]   # (1292, 224, 11)
labels_array = combined_array[:, :, 11:]     # (1292, 224, 1)

print(f'Features: {features_array.shape}, Labels: {labels_array.shape}')
print(f'Mangrove ratio: {(labels_array == 1).mean():.4f}')

# Extract random patches
def extract_patches(features, labels, patch_size=128, n_patches=1000):
    h, w, c = features.shape
    X_list = []
    y_list = []

    np.random.seed(42)
    for i in range(n_patches):
        r = np.random.randint(0, h - patch_size)
        c_idx = np.random.randint(0, w - patch_size)

        patch_x = features[r:r+patch_size, c_idx:c_idx+patch_size]
        patch_y = labels[r:r+patch_size, c_idx:c_idx+patch_size]

        X_list.append(patch_x.transpose(2, 0, 1))  # (C, H, W)
        y_list.append(patch_y.transpose(2, 0, 1))  # (1, H, W)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    # Normalize per channel
    for c in range(X.shape[1]):
        mean = X[:, c].mean()
        std = X[:, c].std() + 1e-8
        X[:, c] = (X[:, c] - mean) / std

    return X, y

X, y = extract_patches(features_array, labels_array, PATCH_SIZE, n_patches=1000)
print(f'X: {X.shape}, y: {y.shape}')
print(f'Mangrove ratio: {y.mean():.4f}')

Features: (1292, 224, 11), Labels: (1292, 224, 1)
Mangrove ratio: 0.0164
X: (1000, 11, 128, 128), y: (1000, 1, 128, 128)
Mangrove ratio: 0.0149


In [14]:
class MangroveDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Split: 70/15/15
n = len(X)
idx = np.random.permutation(n)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_loader = DataLoader(MangroveDataset(X[idx[:n_train]], y[idx[:n_train]]),
                          batch_size=16, shuffle=True)
val_loader = DataLoader(MangroveDataset(X[idx[n_train:n_train+n_val]], y[idx[n_train:n_train+n_val]]),
                        batch_size=16, shuffle=False)
test_loader = DataLoader(MangroveDataset(X[idx[n_train+n_val:]], y[idx[n_train+n_val:]]),
                         batch_size=16, shuffle=False)

print(f'Train: {n_train}, Val: {n_val}, Test: {n - n_train - n_val}')

Train: 700, Val: 150, Test: 150


In [15]:
def compute_iou(preds, targets, threshold=0.5):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    return ((intersection + 1e-7) / (union + 1e-7)).item()

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    total_iou = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            total_loss += criterion(preds, y_batch).item()
            total_iou += compute_iou(preds, y_batch)
    return total_loss / len(loader), total_iou / len(loader)

In [16]:
model = smp.DeepLabV3Plus(
    encoder_name='efficientnet-b0',
    encoder_weights='imagenet',
    in_channels=11,
    classes=1,
).to(device)

# Focal loss handles class imbalance better
criterion = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 15
best_val_iou = 0

print('Training from scratch...')
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_iou = validate(model, val_loader, criterion)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), 'best_model.pt')

    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | IoU: {val_iou:.4f}')

print(f'\nBest Val IoU: {best_val_iou:.4f}')

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Training from scratch...
Epoch 1/15 | Train: 0.0214 | Val: 0.0189 | IoU: 0.2352
Epoch 2/15 | Train: 0.0099 | Val: 0.0080 | IoU: 0.5006
Epoch 3/15 | Train: 0.0061 | Val: 0.0054 | IoU: 0.5188
Epoch 4/15 | Train: 0.0043 | Val: 0.0034 | IoU: 0.5332
Epoch 5/15 | Train: 0.0034 | Val: 0.0028 | IoU: 0.5631
Epoch 6/15 | Train: 0.0029 | Val: 0.0022 | IoU: 0.5972
Epoch 7/15 | Train: 0.0024 | Val: 0.0019 | IoU: 0.6077
Epoch 8/15 | Train: 0.0022 | Val: 0.0016 | IoU: 0.6191
Epoch 9/15 | Train: 0.0019 | Val: 0.0014 | IoU: 0.6246
Epoch 10/15 | Train: 0.0018 | Val: 0.0013 | IoU: 0.6266
Epoch 11/15 | Train: 0.0017 | Val: 0.0013 | IoU: 0.6320
Epoch 12/15 | Train: 0.0016 | Val: 0.0011 | IoU: 0.6329
Epoch 13/15 | Train: 0.0015 | Val: 0.0011 | IoU: 0.6354
Epoch 14/15 | Train: 0.0014 | Val: 0.0010 | IoU: 0.6408
Epoch 15/15 | Train: 0.0014 | Val: 0.0010 | IoU: 0.6454

Best Val IoU: 0.6454


In [17]:
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        preds_binary = (preds > 0.5).float().cpu().numpy()
        all_preds.append(preds_binary)
        all_labels.append(y_batch.numpy())

accuracy = (np.concatenate(all_preds).flatten() == np.concatenate(all_labels).flatten()).mean()
print(f'Test Accuracy: {accuracy:.4f}')

Test Accuracy: 0.9973
